# 32 · Reliability: retries, timeouts, idempotency and rate limits
Level L6 · Part 6 · core · ~25 min · builds on 06, 14

## Goal
Make the refund tool safe to retry. A refund that loses its reply is retried with backoff and jitter, each try has a
timeout, and the money moves once because every retry carries the same idempotency key. A token bucket paces the
requests.

## Why it matters
Lesson 14 made the refund wait for a person. Now the payment API itself fails. A common case: the card is charged,
then the reply never arrives. The client cannot tell "charged" from "not charged", so a plain retry pays again.
A *transient* error is one that may succeed on a retry: a timeout, a lost reply, or a "busy" reply (HTTP 429: the
provider says wait, and nothing was charged). Run the cell below: a retry loop with no key, against a payment stub
whose first reply is lost.

The stub is not a model. It is a fake payment API with scripted faults, so every run fails the same way.

In [1]:
import random
import time
from collections import deque

random.seed(32)  # jitter is random: seed it so every run prints the same numbers
ORDERS = {"1093": 45.00, "2207": 84.00}  # fictional order totals, as in lessons 06 to 14


class ReplyLost(Exception): pass  # the provider charged, but the reply never reached us
class Busy(Exception): pass       # HTTP 429: the provider says wait. Nothing was charged.
class Refused(Exception): pass    # a permanent answer, such as "over the order total"

Each call to the stub below pops one scripted fault (`ok`, `lost`, `slow` or `busy`), so every run fails the same way.
A repeated key returns the saved answer, as Stripe's keys do.

In [2]:
class FlakyPayments:
    def __init__(self, faults=()):
        self.faults, self.calls = deque(faults), 0  # faults to inject; every request the provider saw
        self.charged, self.by_key = [], {}          # the money that really moved; key -> saved answer

    def charge(self, order_id: str, amount_usd: float, idem_key: str | None = None) -> str:
        self.calls += 1
        if idem_key and idem_key in self.by_key:    # same key: the saved answer, and no second charge
            return self.by_key[idem_key]
        if amount_usd > ORDERS[order_id]:
            raise Refused(f"{amount_usd:.2f} USD is over the order total")
        fault = self.faults.popleft() if self.faults else "ok"
        if fault == "busy":
            raise Busy("429 Too Many Requests")
        self.charged.append((order_id, amount_usd))  # the money moves here, before any fault below is raised
        answer = f"charge #{len(self.charged)}"
        if idem_key:
            self.by_key[idem_key] = answer
        if fault == "lost":
            raise ReplyLost("charged, but the reply was lost")
        if fault == "slow":
            time.sleep(0.3)  # the reply comes after the client has given up
        return answer

The loop below has no key. Its first reply is lost after the money moves.

In [3]:
payments = FlakyPayments(faults=["lost"])  # the first reply is lost after the money moves
try:
    payments.charge("2207", 84.00)
except ReplyLost as err:
    print("error:", err)
payments.charge("2207", 84.00)  # the retry: no key, so the provider charges again
print("charges made:", len(payments.charged), "· paid:", sum(a for _, a in payments.charged), "USD for an 84.00 USD order")

error: charged, but the reply was lost
charges made: 2 · paid: 168.0 USD for an 84.00 USD order


**What it makes easy:** a retry that carries the same key returns the first answer, and the card is charged once. The
client can then retry on its own schedule. **When you don't need it:** a read-only lookup changes nothing, so it needs
retries and timeouts but no key.

## The idea
Three rules make a write safe to retry. **Retry only transient errors** (a timeout, a lost reply, a 429), never a
refusal. **Wait longer after each try, with random jitter**, so many clients do not retry together. **Send one
idempotency key** that the code makes from the run id, so the provider can recognise the retry. The key is made by the
code, not by the model: a model may type a different key on the second try.

Analogy: a bank cheque with a serial number. If the teller is unsure whether a cheque was cashed, a second presentation
with the same serial number cannot be cashed twice.

## Picture
```mermaid
sequenceDiagram
  participant C as Refund tool (client)
  participant P as Payment API
  C->>P: charge 84.00, key refund:r-2001
  P-->>P: charges the card (money moves)
  P--xC: reply lost
  Note over C: wait 0.05 s plus jitter
  C->>P: same charge, same key
  P-->>C: saved answer: charge #1 (no second charge)
```

## Already know this?
This is the first lesson of Part 6, the enterprise part. Check these five questions. Each answer is taught in the lesson
named after it, so skip a lesson if you already know its answer.
1. A payment call times out, and the reply is lost after the provider charged the card. What stops a second charge when you retry? → this lesson (32).
2. Caching a long system prompt cuts which part of the bill? → lesson 33 (cost and latency).
3. Three replicas of the agent share one rate limit. Where does the count live? → lesson 34 (serving and deployment).
4. A run that takes 40 minutes dies at minute 38. How does it finish once and only once? → lesson 35 (durable execution).
5. A regulated customer's data must stay in one region. How do tenants and regions get isolated? → lesson 36 (the enterprise map).
If you answer all five, skip to lesson 37, the capstone.

## Step by step
**Step 1 · a timeout and a retry policy (library: tenacity).** Each try gets a timeout, so a slow reply cannot hold the
refund forever. `tenacity` runs the retries, and its `wait_exponential_jitter` adds the backoff and the random part. It
retries only the errors in `TRANSIENT`, and it stops after 3 tries, so each layer makes at most 3 requests. The waits
include random jitter, so they are approximate; the seed in the first code cell makes them repeat.
**Warning:** a timeout does not cancel the call. The charge may still finish in the background, so a timed-out
write needs an idempotency key. Do not copy `call_with_timeout` into a real project without one.

In [4]:
from concurrent.futures import ThreadPoolExecutor

from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential_jitter

POOL = ThreadPoolExecutor(max_workers=4)     # the call runs in a thread, so we can stop waiting for it
TIMEOUT_S = 0.1                              # wait at most 0.1 s for one reply
TRANSIENT = (TimeoutError, ReplyLost, Busy)  # retry these: the money may or may not have moved


def call_with_timeout(fn, *args):
    future = POOL.submit(fn, *args)
    try:
        return future.result(timeout=TIMEOUT_S)
    except TimeoutError:
        raise TimeoutError(f"no reply within {TIMEOUT_S} s") from None


@retry(retry=retry_if_exception_type(TRANSIENT),                # only transient errors
       stop=stop_after_attempt(3),                              # 3 tries in all
       wait=wait_exponential_jitter(initial=0.05, max=0.5),     # starts at 0.05 s, plus up to 1 s of jitter, capped at 0.5 s
       reraise=True)                                            # after the last try, raise the real error
def pay_refund(payments, order_id: str, amount_usd: float, idem_key: str) -> str:
    return call_with_timeout(payments.charge, order_id, amount_usd, idem_key)

**Step 2 · the drill (no model).** Two runs. In the first, the reply is lost after the charge. In the second, the
provider first says "busy" (429), then its reply is slow, past the timeout. Trace run 2: the "busy" try charges nothing,
so the client retries. The "slow" try charges the card in the background, and the client gives up after 0.1 s. The next
try carries the same key, so it gets the saved answer and the card is not charged again. Both runs use one key per refund.

In [5]:
payments = FlakyPayments(faults=["lost"])
print("1. lost reply:", pay_refund(payments, "2207", 84.00, idem_key="refund:r-2001"))
print("   requests:", payments.calls, "· charges made:", len(payments.charged))

payments = FlakyPayments(faults=["busy", "slow"])
print("2. busy, then slow:", pay_refund(payments, "2207", 84.00, idem_key="refund:r-2002"))
print("   requests:", payments.calls, "· charges made:", len(payments.charged))

1. lost reply: charge #1
   requests: 2 · charges made: 1


2. busy, then slow: charge #1
   requests: 3 · charges made: 1


**Step 3 · the agent asks for the refund (needs-llm).** The approved refund from lesson 14 is paid by the tool. The model
is given one tool, `issue_refund`, and it makes one call. This is the only model call of the notebook. The SDK retries
busy-server errors (429, 5xx) on its own, as in lesson 11.

In [6]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("32_reliability-retries-idempotency")

from google import genai  # the official Google SDK
from google.genai import types
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"  # the course's default model
client = genai.Client(
    vertexai=True,
    api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
    http_options=types.HttpOptions(timeout=30_000, retry_options=types.HttpRetryOptions(
        attempts=3, initial_delay=5.0, max_delay=30.0)),  # the SDK's own retry for 429 and 5xx: 3 tries, as above
)

LLM mode: replay (32_reliability-retries-idempotency.yaml)


The model sees one tool, `issue_refund`, and a customer message that says the refund is approved.

In [7]:
ISSUE = types.FunctionDeclaration(
    name="issue_refund", description="Pay an approved refund for one order.",
    parameters_json_schema={"type": "object", "properties": {"order_id": {"type": "string"},
                            "amount_usd": {"type": "number"}}, "required": ["order_id", "amount_usd"]})
config = types.GenerateContentConfig(
    system_instruction="You are the support assistant for Brightside Outfitters. The refund is approved: call issue_refund.",
    tools=[types.Tool(function_declarations=[ISSUE])], temperature=0)
reply = client.models.generate_content(  # the only model call of this notebook
    model=MODEL, contents="Order 2207 arrived with a torn box. The refund of 84.00 USD is approved. Please issue it.",
    config=config)
llm_replay.stop()
call = next(p.function_call for p in reply.candidates[0].content.parts if p.function_call)
args = dict(call.args)
print("model asked:", call.name, args)

model asked: issue_refund {'order_id': '2207', 'amount_usd': 84}


**Fallback (only if the model cell was skipped).** The same arguments are typed in by hand, as the model would send
them. It is not model output, and the rest of the notebook still runs.

In [8]:
args = globals().get("args", {"order_id": "2207", "amount_usd": 84.00})  # the fallback when the model cell was skipped
print("arguments used:", args)

arguments used: {'order_id': '2207', 'amount_usd': 84}


The code makes the key from the run id, `refund:r-2003`, and then runs the tool. The drill has a busy reply and then a
lost reply. The model's arguments are data, not trusted values. The recording sends `amount_usd` as the integer 84, so
the code converts both fields and checks the order before it charges (lesson 04 covers validation).

In [9]:
payments = FlakyPayments(faults=["busy", "lost"])
order_id, amount = str(args["order_id"]), float(args["amount_usd"])  # convert the model's arguments first
assert order_id in ORDERS and amount > 0, "refusing to charge an unknown order or a bad amount"
answer = pay_refund(payments, order_id, amount, idem_key="refund:r-2003")
print("paid:", answer, "· requests:", payments.calls, "· charges made:", len(payments.charged))

paid: charge #1 · requests: 3 · charges made: 1


## Under the hood
The fake payment API above is a test stub, not a retry mechanism: it only scripts the faults. The token bucket is the
hand-coded interview item. No standard library in this stack limits request rates, so the bucket is written by hand. It
is the interview version of a rate limiter. The bucket holds `capacity` tokens and refills at `rate` tokens per second.
Each request spends one token. When the bucket is empty, the request must wait, and the bucket tells the caller how long.
Times are passed in, so the demo is exact and fast. The balance may go negative: the bucket lends the token, and the
caller waits for the debt to be repaid at `rate` per second.

In [10]:
class TokenBucket:
    def __init__(self, capacity: float, rate: float):
        self.capacity, self.rate = capacity, rate
        self.tokens, self.last = capacity, 0.0  # starts full

    def take(self, now: float) -> float:
        """Spend one token. Return the seconds the caller must wait first (0.0 = send now)."""
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)  # refill since the last call
        self.last = now
        self.tokens -= 1.0                       # reserve the token, even if the bucket goes below zero
        return max(0.0, -self.tokens / self.rate)


bucket = TokenBucket(capacity=2, rate=2.0)     # bursts of 2, then 2 requests per second
print("five requests at t = 0 s, waits:", [bucket.take(now=0.0) for _ in range(5)])
# balance after each take: 1, 0, -1, -2, -3; wait = -balance / rate gives 0, 0, 0.5, 1.0, 1.5 s
print("a request at t = 2 s, wait:", bucket.take(now=2.0))  # refill 4, capped at 2: -3 -> 1 -> 0, wait 0

five requests at t = 0 s, waits: [0.0, 0.0, 0.5, 1.0, 1.5]
a request at t = 2 s, wait: 0.0


## At real scale
- **Jitter stops the herd.** With jitter and 100 contending clients, the
  call count fell by more than half ([Brooker, AWS Architecture Blog, 2015](https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/)).
- **Retries multiply, and replicas share the bucket.** Three layers that each retry three times send 27 requests for one
  user action. Three replicas, each with its own bucket, allow three times the rate. Retry in one layer only and cap the
  total (→ lesson 33). Keep the count in a shared store behind the gateway (→ lesson 34).
- **Keys have a life.** Stripe returns the saved answer for a repeat, and deletes a key only after 24 hours. Keys hold
  up to 255 characters; keep personal data out ([Stripe API docs](https://docs.stripe.com/api/idempotent_requests)).
- **Go deeper:** circuit breakers, `Retry-After`, fault injection with a proxy (toxiproxy), run deadlines (lesson 35).

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| Read-only lookup (order status) | retry 429 and 5xx, timeout, no key | a repeat changes nothing |
| Write that moves money | one key from the run id, retry only transient errors | the repeat returns the saved answer |
| Refusal ("over the order total") | no retry, return the error to the agent | retrying repeats the refusal |
| Many users at once | token bucket per tenant or key, honour `Retry-After` | stops 429 storms before they start |
| Run longer than one request | run deadline and a durable run (lesson 35) | a retry cannot span a crash |

## Common mistakes
1. **Retrying every error.** This decorator retries the refusal too. The provider sees the same refused request three
   times. The fix is `retry_if_exception_type(TRANSIENT)`, as in Step 1.

In [11]:
from tenacity import retry as naive_retry  # the wrong policy: no filter on the error type

p = FlakyPayments()


@naive_retry(stop=stop_after_attempt(3), reraise=True)  # retries everything, including a refusal
def refund_everything(payments):
    return payments.charge("2207", 500.00, "refund:r-2004")  # over the order total


refund_everything(p)

Refused: 500.00 USD is over the order total

The raise above ended the cell on purpose. The next cell counts what the provider saw.

In [12]:
print("requests the provider saw:", p.calls)  # 3: the refusal was sent three times, then the retries stopped

requests the provider saw: 3


The refusal went out three times, as the count shows. A permanent answer should reach the agent once, with its reason.

2. **No key on the retry.** Without the key the second try is a new charge (the first cell of Step 2 shows it). Keep one
   key per refund, made from the run id.
3. **Retries with no jitter.** Every client that failed together waits the same time and retries together. Use
   `wait_exponential_jitter`, not `wait_exponential`.

## Try it
Predict the five waits for `TokenBucket(capacity=3, rate=1.0)` with five requests at t = 0 s. Then check the prediction
with the code. Then predict the wait for one request at t = 1.5 s.

In [13]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [14]:
bucket = TokenBucket(capacity=3, rate=1.0)  # predicted: [0.0, 0.0, 0.0, 1.0, 2.0]
print("five requests at t = 0 s, waits:", [bucket.take(now=0.0) for _ in range(5)])
# the balance after five requests is -2.0; at t = 1.5 s, 1.5 tokens refill (-0.5), and one more request makes it -1.5
print("a request at t = 1.5 s, wait:", bucket.take(now=1.5))  # predicted: 1.5

five requests at t = 0 s, waits: [0.0, 0.0, 0.0, 1.0, 2.0]
a request at t = 1.5 s, wait: 1.5


## Recap
- A timeout bounds each try. Tenacity retries only transient errors, with backoff and jitter, and stops after 3 tries.
- A write carries one idempotency key from the run id. A repeat returns the saved answer, so the money moves once.
- A refusal is permanent: return it to the agent, do not retry.
- A token bucket allows bursts and paces the rest. At scale, the count lives in a shared store behind the gateway.

**Interview:** "The payment API times out after it charged the card. How do you retry safely?"
Strong answer: "Retry only timeouts and 429s, with exponential backoff and jitter, and send the same idempotency key every
time, made from the run id. The provider returns the saved answer on a repeat, so the card is charged once."